# Day 65: Convolutional Neural Networks (CNNs)
## How Computers Learn to "See"

---

# PART 1: THEORY

## 1. Why Dense Networks Fail for Images

A 200x200 RGB image has 120,000 pixels. A dense layer with 128 neurons = **15.3 million parameters** — just for the FIRST layer!

**Three problems with Dense layers for images:**
1. **Too many parameters** -> overfitting, slow training
2. **Ignores spatial structure** — a cat on the left vs right are completely different inputs to a Dense network
3. **No translation invariance** — the network has to learn "cat" separately for every position

**CNNs solve all three problems.**

## 2. The Core Idea — Convolution

Instead of connecting every input to every neuron, slide a **small filter** (kernel) across the image:

```
Image:   [1  2  3]     Filter: [1  0]
         [4  5  6]             [-1 1]
         [7  8  9]

Slide filter across image -> multiply element-wise -> sum -> Feature Map
```

**Each filter detects a pattern:**
- Some filters detect horizontal edges
- Some detect vertical edges
- Some detect corners, textures, colors
- Deeper filters detect faces, eyes, wheels, etc.

## 3. CNN Building Blocks

| Layer | What It Does | Example |
|-------|-------------|---------|
| **Conv2D** | Apply filters, detect features | Conv2D(32, 3x3) = 32 filters of size 3x3 |
| **MaxPool2D** | Downsample, reduce dimensions | MaxPool(2x2) halves height and width |
| **Flatten** | Convert 2D feature maps to 1D vector | Connect to Dense layers |
| **Dropout** | Regularization | Randomly disable neurons |

## 4. A Typical CNN Architecture

```
Input (32x32x3)
  -> Conv2D(32, 3x3) + ReLU  -> 30x30x32
  -> MaxPool(2x2)             -> 15x15x32
  -> Conv2D(64, 3x3) + ReLU   -> 13x13x64
  -> MaxPool(2x2)             -> 6x6x64
  -> Conv2D(128, 3x3) + ReLU  -> 4x4x128
  -> MaxPool(2x2)             -> 2x2x128
  -> Flatten                  -> 512
  -> Dense(128) + Dropout
  -> Dense(10) + Softmax
```

**Pattern:** Features increase (32->64->128), spatial size decreases (32->16->8).

---

# PART 2: PRACTICAL

## 5. Load CIFAR-10

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train: {X_train.shape} — {X_train.shape[0]:,} images, {X_train.shape[1]}x{X_train.shape[2]}, RGB")
print(f"Test:  {X_test.shape}")
print(f"Classes: {len(classes)}")
print(f"Pixel range: [{X_train.min()}, {X_train.max()}]")


In [ ]:
# Show samples from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(y_train.flatten() == i)[0][0]
    ax.imshow(X_train[idx])
    ax.set_title(classes[i], fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('CIFAR-10 — One Sample Per Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Preprocess and Build CNN

In [ ]:
# Normalize
X_train_n = X_train.astype('float32') / 255.0
X_test_n = X_test.astype('float32') / 255.0

# Build CNN
cnn = keras.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Classifier
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

cnn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()


## 7. Train the CNN

In [ ]:
# Train with callbacks
early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

history = cnn.fit(X_train_n, y_train, epochs=30, batch_size=64,
                  validation_split=0.1,
                  callbacks=[early_stop, reduce_lr], verbose=1)


In [ ]:
# Plot training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CNN Training Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('CNN Training Loss')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()


## 8. Evaluate and Compare with Dense Network

In [ ]:
# Evaluate CNN
cnn_loss, cnn_acc = cnn.evaluate(X_test_n, y_test, verbose=0)
print(f"CNN Test Accuracy: {cnn_acc:.3f} ({cnn_acc*100:.1f}%)")

# Compare with Dense network (flatten image)
dense_model = keras.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
dense_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
dense_model.fit(X_train_n, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
_, dense_acc = dense_model.evaluate(X_test_n, y_test, verbose=0)

print(f"\nCNN:   {cnn_acc:.3f} ({cnn.count_params():,} params)")
print(f"Dense: {dense_acc:.3f} ({dense_model.count_params():,} params)")
print(f"\nCNN is both MORE accurate AND has FEWER parameters!")


In [ ]:
# Visualize predictions
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred = np.argmax(cnn.predict(X_test_n, verbose=0), axis=1)

# Show some predictions
fig, axes = plt.subplots(3, 5, figsize=(14, 9))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(0, len(X_test))
    ax.imshow(X_test[idx])
    true_label = classes[y_test[idx][0]]
    pred_label = classes[y_pred[idx]]
    color = 'green' if y_pred[idx] == y_test[idx][0] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('CNN Predictions — Green=Correct, Red=Wrong', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---

# PART 3: EXERCISES

In [ ]:
# Exercise 1: Try Data Augmentation
data_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Apply augmentation to a sample image
sample = X_train[0]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax in axes:
    augmented = data_aug(tf.expand_dims(sample, 0), training=True)
    ax.imshow(augmented[0].numpy().astype('int'))
    ax.axis('off')
plt.suptitle('Data Augmentation — Same Image, Different Views', fontsize=14)
plt.show()


In [ ]:
# Exercise 2: Try different filter counts
# Compare: 16-32-64 vs 32-64-128 vs 64-128-256
for filters in [(16, 32, 64), (32, 64, 128), (64, 128, 256)]:
    m = keras.Sequential([
        layers.Conv2D(filters[0], 3, activation='relu', padding='same', input_shape=(32,32,3)),
        layers.MaxPooling2D(2),
        layers.Conv2D(filters[1], 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(filters[2], 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    m.fit(X_train_n, y_train, epochs=5, validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_test_n, y_test, verbose=0)
    print(f"Filters {str(filters):20s} -> Test Acc: {acc:.3f} | Params: {m.count_params():,}")


## Key Takeaways

- **CNNs** use convolution to detect spatial patterns in images
- **Filters** slide across the image detecting edges, textures, shapes
- **Pooling** reduces dimensions and provides translation invariance
- CNNs have **far fewer parameters** than Dense networks for images
- **Deeper layers** learn more abstract features
- **Data augmentation** creates more training data artificially
- CIFAR-10 is the classic benchmark — CNN achieves 75-85% accuracy

**Tomorrow:** Transfer Learning — use pre-trained models from ImageNet!